In [3]:
# Bloco do ponto de montagem do google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Bloco de importação de bibliotecas
import pandas as pd
import numpy as np
import random
import os
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [5]:
# RF01 – Criar ou Carregar o Dataset de Vendas
def gerar_dataset_vendas(n_registros=150, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse']
    precos = {'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800, 'Monitor': 1200, 'Teclado': 250, 'Mouse': 120}
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares", "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos", "Mouse": "Periféricos"}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    data_inicio = datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo sujeira intencional
        if random.random() < 0.05: quantidade = None
        if random.random() < 0.04: preco = None
        if random.random() < 0.03: produto = "   " + produto + "  "

        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

# 1. Definindo o caminho exato para o drive na pasta DataView-Vendas
CAMINHO_PROJETO = "/content/drive/MyDrive/Pessoal/Cursos/SCTEC/PROFISSIONALIZAR/Desenvolvimento de IA para Análise Preditiva/Fundamentos de Dados, Programação e Análise Preditiva com Python [T2]/MINIPROJETO/DataView-Vendas"

# 2. Executando a função para gerar o dataset sintético bruto
df_bruto = gerar_dataset_vendas()

# 3. Mapeando e criando a estrutura 'data/raw' dentro da sua pasta do Drive
caminho_raw = os.path.join(CAMINHO_PROJETO, "data/raw")
os.makedirs(caminho_raw, exist_ok=True)

# 4. Salvando o arquivo vendas.csv bruto diretamente no Drive
caminho_arquivo_final = os.path.join(caminho_raw, "vendas.csv")
df_bruto.to_csv(caminho_arquivo_final, index=False)

print("=== SUCESSO TRILHADO ===")
print(f"Dataset bruto gerado com {len(df_bruto)} registros.")
print(f"Salvo no Google Drive em:\n{caminho_arquivo_final}")

=== SUCESSO TRILHADO ===
Dataset bruto gerado com 150 registros.
Salvo no Google Drive em:
/content/drive/MyDrive/Pessoal/Cursos/SCTEC/PROFISSIONALIZAR/Desenvolvimento de IA para Análise Preditiva/Fundamentos de Dados, Programação e Análise Preditiva com Python [T2]/MINIPROJETO/DataView-Vendas/data/raw/vendas.csv


In [6]:
# RF02 – Inspecionar e Descrever os Dados
def inspecionar_dados(df):
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados: \n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    return df.describe(include="all")

inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (150, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados: 
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        5
preco_unitario    2
dtype: int64


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.000000,150,150,150,150,150,145.000000,148.000000
unique,NaN,117,30,8,3,5,NaN,NaN
top,NaN,DATA INVALIDA,Cliente_018,Mouse,Celulares,Sudeste,NaN,NaN
freq,NaN,4,8,28,51,41,NaN,NaN
mean,75.500000,NaN,NaN,NaN,NaN,NaN,5.468966,1558.513514
std,43.445368,NaN,NaN,NaN,NaN,NaN,2.808853,1190.199414
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,120.000000
25%,38.250000,NaN,NaN,NaN,NaN,NaN,3.000000,250.000000
50%,75.500000,NaN,NaN,NaN,NaN,NaN,5.000000,1800.000000
75%,112.750000,NaN,NaN,NaN,NaN,NaN,8.000000,2200.000000


In [7]:
# RF03 – Limpar e Tratar os Dados
# Função 1: Limpeza de strings com Regex
def limpar_strings_regex(df, colunas):
    """Remove espaços duplos e espaços nas pontas das strings."""
    df = df.copy()
    for col in colunas:
        df[col] = df[col].apply(
            lambda s: re.sub(r"\s+", " ", str(s)).strip() if pd.notna(s) else s
        )
    return df

# Função 2: Fluxo principal de limpeza
def limpar_dados(df):
    df = df.copy()
    n_inicial = len(df)
    relatorio = {}

    # 1. Limpeza de strings (apenas colunas de texto)
    colunas_texto = df.select_dtypes(include="object").columns
    df = limpar_strings_regex(df, colunas_texto)

    # 2. Conversão de datas e remoção das inválidas ("DATA INVALIDA" vira NaT e é dropada)
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    relatorio["datas_invalidas_removidas"] = int(df["data_venda"].isnull().sum())
    df = df.dropna(subset=["data_venda"])

    # 3. Remoção de nulos nas colunas críticas
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    # 4. Ajuste de tipos numéricos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # Fechamento do relatório
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = n_inicial - len(df)

    print("=== RELATÓRIO DE LIMPEZA ===")
    for k, v in relatorio.items():
        print(f" {k}: {v}")

    return df, relatorio

# --- EXECUÇÃO E EXPORTAÇÃO DA V1 ---
df_v1, relatorio = limpar_dados(df_bruto)

# Cria a pasta e salva no Google Drive (usando a variável CAMINHO_PROJETO definida na RF01)
caminho_v1 = os.path.join(CAMINHO_PROJETO, "data/processed/v1_com_outliers")
os.makedirs(caminho_v1, exist_ok=True)

caminho_arquivo_v1 = os.path.join(caminho_v1, "vendas_v1.csv")
df_v1.to_csv(caminho_arquivo_v1, index=False)

print(f"\n✅ Versão V1 salva com sucesso em:\n{caminho_arquivo_v1}")

=== RELATÓRIO DE LIMPEZA ===
 datas_invalidas_removidas: 4
 linhas_nulas_removidas: 6
 registros_iniciais: 150
 registros_finais: 140
 registros_removidos_total: 10

✅ Versão V1 salva com sucesso em:
/content/drive/MyDrive/Pessoal/Cursos/SCTEC/PROFISSIONALIZAR/Desenvolvimento de IA para Análise Preditiva/Fundamentos de Dados, Programação e Análise Preditiva com Python [T2]/MINIPROJETO/DataView-Vendas/data/processed/v1_com_outliers/vendas_v1.csv
